In [ ]:
! pip install --upgrade -q pandas numpy scikit-learn imbalanced-learn optuna

In [ ]:
from collections import Counter
from pathlib import Path

import pandas as pd
import numpy as np

from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split, StratifiedKFold
from sklearn.metrics import f1_score
from sklearn.ensemble import RandomForestClassifier
from sklearn.compose import ColumnTransformer

from imblearn.over_sampling import SMOTE
from imblearn.under_sampling import RandomUnderSampler
from imblearn.pipeline import Pipeline

import optuna

from utils.constants import *

In [ ]:
output_dir = Path("../results/hp_tuning/random_forest")
output_dir.mkdir(exist_ok=True)

df = pd.read_csv("../data/3_gold/dataset-processed-continuous-onehot.csv")

X = df.drop("class", axis=1)
y = df["class"]
y = y.map(TARGET_LABEL_MAP)

# numeric_indices = [X.columns.get_loc(c) for c in NUMERIC_COLUMNS if c in X.columns]

feature_names = X.columns.tolist()
print(feature_names)

In [ ]:
class_counts = Counter(y)
for class_idx, count in class_counts.items():
    print(f"Number of '{TARGET_NAMES[class_idx]}' examples: {count}")

In [ ]:
X_train_full, X_test, y_train_full, y_test = train_test_split(
    X, y, test_size=TEST_RATIO, random_state=RANDOM_STATE, stratify=y
)

In [ ]:
def objective(trial):
    params = {
        "n_estimators": trial.suggest_int("n_estimators", 100, 500, step=50),
        "max_depth": trial.suggest_int("max_depth", 5, 50),
        "min_samples_split": trial.suggest_int("min_samples_split", 2, 20),
        "min_samples_leaf": trial.suggest_int("min_samples_leaf", 1, 10),
        "max_features": trial.suggest_categorical("max_features", ["sqrt", "log2"]),
        "class_weight": "balanced",
        "random_state": RANDOM_STATE,
        "n_jobs": -1
    }

    skf = StratifiedKFold(n_splits=N_FOLDS, shuffle=True, random_state=RANDOM_STATE)
    scores = []

    for train_idx, valid_idx in skf.split(X_train_full, y_train_full):
        X_train, y_train = X_train_full.iloc[train_idx], y_train_full.iloc[train_idx]
        X_valid, y_valid = X_train_full.iloc[valid_idx], y_train_full.iloc[valid_idx]

        steps = []
        
        # --- Standard Scaling ---
        # if numeric_indices:
        #     preprocessor = ColumnTransformer(
        #         transformers=[('num', StandardScaler(), NUMERIC_COLUMNS)],
        #         remainder='passthrough' # Keep binary cols as is
        #     )
        #     steps.append(('preprocessor', preprocessor))
        
        # --- Resampling ---
        counts = Counter(y_train)
        n_alarm = counts[1]

        steps.append(('under', RandomUnderSampler(sampling_strategy={0: max(n_alarm, counts[0]//2)}, random_state=RANDOM_STATE)))
        steps.append(('over', SMOTE(sampling_strategy={2: n_alarm}, random_state=RANDOM_STATE)))
        
        steps.append(('model', RandomForestClassifier(**params)))
        pipeline = Pipeline(steps)

        # --- Fit & Evaluate ---
        try:
            pipeline.fit(X_train, y_train)
            preds = pipeline.predict(X_valid)
            f1 = f1_score(y_valid, preds, average='macro')
            scores.append(f1)
        except ValueError:
            return 0.0

    return np.mean(scores)

In [ ]:
study = optuna.create_study(direction="maximize")
study.optimize(objective, n_trials=50, timeout=14400, show_progress_bar=True, n_jobs=-1)

best_trial = study.best_trial
print("Best F1:", best_trial.value)
print("Best Params:", best_trial.params)

In [ ]:
import pickle

study_path = output_dir / "optuna_study.pkl"
with open(study_path, "wb") as f:
    pickle.dump(study, f)

In [ ]:
! pip install plotly nbformat

In [ ]:
import optuna.visualization as vis
import pickle

study_path = output_dir / "optuna_study.pkl"
with open(study_path, 'rb') as f:
    study = pickle.load(f)

display(vis.plot_param_importances(study))
display(vis.plot_optimization_history(study))